# 05 — Evaluation

Compares three systems on 200 health-focused evaluation queries across 4 dietary categories:
- Diabetic-friendly (50 queries)
- High-protein / Workout (50 queries)
- Gluten-free (50 queries)
- Low-fat / Heart health (50 queries)

**Metrics:**
| Metric | What it measures |
|--------|------------------|
| Recall@5/10 | Fraction of relevant recipes found in top-K |
| MRR | How high is the first relevant result? |
| NDCG@10 | Are the highest-rated recipes ranked first? |
| Nutritional Precision@5 | Fraction of top-5 that actually satisfy the dietary constraint |

**Ground truth construction:**
Relevant recipes = tagged with target dietary flag AND mean_rating ≥ 4.0 AND review_count ≥ 10.
This uses actual community ratings from 1.1M interactions — not manual annotation.

In [ ]:
import os
os.chdir('..')  # run from project root so all data/ paths resolve correctly

import sys
sys.path.insert(0, '.')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Run Evaluation (all 3 systems)

In [ ]:
from src.evaluation.evaluator import run_evaluation
results = run_evaluation(config_path='configs/config.yaml')
print(results.to_string())

## 2. Metrics Comparison Bar Chart

In [ ]:
metrics_to_plot = ['recall@5', 'recall@10', 'mrr', 'ndcg@10', 'nutritional_precision@5']
plot_data = results[metrics_to_plot]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(18, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for ax, metric in zip(axes, metrics_to_plot):
    bars = ax.bar(plot_data.index, plot_data[metric], color=colors, edgecolor='white', width=0.6)
    ax.set_title(metric.upper(), fontsize=11, fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.set_xticklabels(plot_data.index, rotation=15, ha='right')
    for bar, val in zip(bars, plot_data[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Retrieval System Comparison — 200 Dietary Queries', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to data/processed/evaluation_results.png')

## 3. Key Findings

Interpret the results in plain language.

In [ ]:
bm25 = results.loc['BM25']
full = results.loc['Full Pipeline']

recall_lift = (full['recall@10'] - bm25['recall@10']) / bm25['recall@10'] * 100
mrr_lift = (full['mrr'] - bm25['mrr']) / bm25['mrr'] * 100
nut_prec_lift = (full['nutritional_precision@5'] - bm25['nutritional_precision@5']) / bm25['nutritional_precision@5'] * 100

print('Key findings:')
print(f'  Recall@10 improvement over BM25:            +{recall_lift:.0f}%')
print(f'  MRR improvement over BM25:                  +{mrr_lift:.0f}%')
print(f'  Nutritional Precision@5 improvement:        +{nut_prec_lift:.0f}%')
print()
print('Interpretation:')
print('  - Semantic embeddings capture meaning that keyword search misses:')
print('    "post-workout meal" matches "high-protein chicken bowl" even with no keyword overlap.')
print('  - The nutritional re-ranker provides the biggest gain on Nutritional Precision:')
print('    it ensures that results actually satisfy the dietary constraint, not just sound related.')
print('  - The domain-specific Nutritional Precision@5 metric is the most important for')
print('    a health-oriented system — a recipe that sounds diabetic-friendly but has high')
print('    glycemic load is actively harmful to the user.')

## 4. Per-Category Breakdown

In [ ]:
# This cell shows how results differ across the 4 dietary categories
# (requires per-category results to be exposed from evaluator — shows the concept)

category_names = ['Diabetic', 'High-Protein', 'Gluten-Free', 'Low-Fat']
# Placeholder values — replace with actual per-category eval output
nut_prec_full = [0.85, 0.88, 0.79, 0.82]
nut_prec_bm25 = [0.42, 0.48, 0.44, 0.46]

x = range(len(category_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], nut_prec_bm25, width, label='BM25', color='#e74c3c', alpha=0.8)
ax.bar([i + width/2 for i in x], nut_prec_full, width, label='Full Pipeline', color='#2ecc71', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(category_names)
ax.set_ylabel('Nutritional Precision@5')
ax.set_title('Nutritional Precision@5 by Dietary Category', fontsize=13)
ax.legend()
ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.show()